# Lab 2 - Auditing and fixing an AI compliance plan

**Objective.** Detect gaps in a fictional compliance plan, link each gap to a corrective action, and generate a traceable Markdown plan.

The control below is deliberately simple and rule-based. In a real setting, rules must be validated by legal, security, data, and business teams; they do not replace regulatory analysis.

## Working method

The notebook starts from a flawed plan, applies transparent detection rules, then creates a corrective document. For every finding, check three things: does the finding match the text? does the correction address the real cause? what evidence will demonstrate that the control was applied? The tests validate the program's behaviour, not an organisation's effective legal compliance.

In [1]:
from pathlib import Path
from datetime import date
import re

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.txt").exists()), Path.cwd())
ARTIFACTS = PROJECT_ROOT / "notebooks_en" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

## 1. Define a deliberately incomplete plan

Each item is a statement to audit. The examples use the lab's themes: consent, bias testing, and logging. All data are fictional.

**Why start with this format?** A list makes each statement individually testable. In a real audit, it would come from procedures, contracts, impact assessments, and technical evidence. The three sentences are deliberately negative: they are the inputs that the mechanism must detect.

In [2]:
broken_checks = [
    "Data is collected without user consent.",
    "Bias testing is skipped because of time constraints.",
    "No predictions are logged.",
]
for item in broken_checks:
    print(f"[!] {item}")


[!] Data is collected without user consent.
[!] Bias testing is skipped because of time constraints.
[!] No predictions are logged.


In [3]:
# Test - the scenario covers the three expected gap categories.
assert len(broken_checks) == 3
text = " ".join(broken_checks).lower()
assert all(token in text for token in ("consent", "bias", "logged"))
print("OK - audit scenario initialised.")


OK - audit scenario initialised.


## 2. Detect non-compliance using explicit rules

A rule must be readable and testable. The function returns a list of findings instead of a simple Boolean, making the audit explainable and traceable.

**Function flow:** `RULES` links every topic to a search pattern, a domain, a severity, and a correction. `audit_plan` loops through each statement and then each rule; when a pattern matches, it creates a finding dictionary. The output therefore retains both the original text and the recommendation. Regular expressions suit this example; a real audit would use controlled vocabulary and human review of false positives and false negatives.

In [4]:
RULES = {
    "consent": {"pattern": r"without user consent", "domain": "Legal", "severity": "High", "correction": "Obtain and retain evidence of valid consent before collection."},
    "fairness": {"pattern": r"bias testing.*skipped", "domain": "Ethical", "severity": "High", "correction": "Plan fairness metrics, thresholds, and an independent review before production."},
    "traceability": {"pattern": r"no predictions are logged", "domain": "Operational", "severity": "High", "correction": "Log versions, decisions, and audit-relevant events using data minimisation."},
}

def audit_plan(items, rules=RULES):
    findings = []
    for item in items:
        for rule_id, rule in rules.items():
            if re.search(rule["pattern"], item, flags=re.IGNORECASE):
                findings.append({"rule": rule_id, "finding": item, **{key: value for key, value in rule.items() if key != "pattern"}})
    return findings

findings = audit_plan(broken_checks)
findings


[{'rule': 'consent',
  'finding': 'Data is collected without user consent.',
  'domain': 'Legal',
  'severity': 'High',
  'correction': 'Obtain and retain evidence of valid consent before collection.'},
 {'rule': 'fairness',
  'finding': 'Bias testing is skipped because of time constraints.',
  'domain': 'Ethical',
  'severity': 'High',
  'correction': 'Plan fairness metrics, thresholds, and an independent review before production.'},
 {'rule': 'traceability',
  'finding': 'No predictions are logged.',
  'domain': 'Operational',
  'severity': 'High',
  'correction': 'Log versions, decisions, and audit-relevant events using data minimisation.'}]

In [5]:
# Test - every gap triggers exactly one finding with a correction.
assert len(findings) == len(broken_checks)
assert {finding["rule"] for finding in findings} == set(RULES)
assert all(finding["correction"] and finding["domain"] for finding in findings)
print("OK - all expected rules were detected.")


OK - all expected rules were detected.


## 3. Generate the corrected plan

The document links a finding to its domain, severity, and recommended control. Its date is generated automatically so that an example date is not mistaken for real information.

**Production steps:** `render_plan` converts the finding list into Markdown; each action receives a title and structured fields; `write_text` saves the result in UTF-8 to preserve accents; and `read_text` displays it for immediate review. The Test cell then opens the actual saved file and verifies its heading, findings, and corrections - it does not merely test the in-memory Python object.

In [6]:
def render_plan(findings, owner="AI Compliance Team"):
    lines = ["# Revised AI Compliance Plan", "", "## Metadata", f"- **Owner:** {owner}", f"- **Generation date:** {date.today().isoformat()}", "- **Status:** Pending validation by the relevant functions", ""]
    for number, finding in enumerate(findings, start=1):
        lines += [f"## Action {number} - {finding['rule'].title()}", f"- **Finding:** {finding['finding']}", f"- **Domain:** {finding['domain']}", f"- **Severity:** {finding['severity']}", f"- **Recommended control:** {finding['correction']}", "- **Expected evidence:** approved procedure, control result, and named owner.", ""]
    return "\n".join(lines)

plan = render_plan(findings)
output = ARTIFACTS / "lab2_revised_compliance_plan.md"
output.write_text(plan, encoding="utf-8")
print(output.read_text(encoding="utf-8"))


# Revised AI Compliance Plan

## Metadata
- **Owner:** AI Compliance Team
- **Generation date:** 2026-11-02
- **Status:** Pending validation by the relevant functions

## Action 1 - Consent
- **Finding:** Data is collected without user consent.
- **Domain:** Legal
- **Severity:** High
- **Recommended control:** Obtain and retain evidence of valid consent before collection.
- **Expected evidence:** approved procedure, control result, and named owner.

## Action 2 - Fairness
- **Finding:** Bias testing is skipped because of time constraints.
- **Domain:** Ethical
- **Severity:** High
- **Recommended control:** Plan fairness metrics, thresholds, and an independent review before production.
- **Expected evidence:** approved procedure, control result, and named owner.

## Action 3 - Traceability
- **Finding:** No predictions are logged.
- **Domain:** Operational
- **Severity:** High
- **Recommended control:** Log versions, decisions, and audit-relevant events using data minimisation.
- **Ex

In [7]:
# Test - the artefact is created, non-empty, and contains every finding and correction.
assert output.exists() and output.stat().st_size > 300
saved = output.read_text(encoding="utf-8")
assert saved.startswith("# Revised AI Compliance Plan")
for finding in findings:
    assert finding["finding"] in saved
    assert finding["correction"] in saved
print("OK - complete, traceable Markdown plan.")


OK - complete, traceable Markdown plan.


## 4. Limits and next audit steps

The tests confirm that the detection mechanism and document are consistent. They do not show that the controls are applied in practice. The next step is to assign a due date, collect evidence, control access to logs, and have the plan approved by the appropriate owners.